# 05 • Resumir dados com groupby

## Preparar esta sessão
Execute as células abaixo antes de começar a aula. Elas localizam os arquivos e preparam os objetos necessários. Você não precisa decorar essa preparação agora.

In [ ]:
from pathlib import Path
import pandas as pd
import math
BASE = Path.cwd()
if not (BASE / 'dados').exists() and (BASE.parent / 'dados').exists():
    BASE = BASE.parent
DADOS = BASE / 'dados'
SAIDAS = BASE / 'resultados'
SAIDAS.mkdir(exist_ok=True)
assert (DADOS/'penguins.csv').exists(), 'Descompacte o curso inteiro e abra na pasta correta.'


## Preparação independente
Execute esta célula para recriar os objetos necessários dos módulos anteriores.

In [ ]:
p = pd.read_csv(DADOS / 'penguins.csv')
print(p.head(3).to_string(index=False))
print('Dimensões:', p.shape)
print('Colunas:', p.columns.tolist())
print(type(p['species']).__name__)

mascara = p['species'].eq('Adelie') & p['body_mass_g'].gt(4000)
print(p.loc[mascara, ['species', 'island', 'body_mass_g']].head())
print(p.iloc[:3, :2])

qualidade = pd.DataFrame({'ausentes':p.isna().sum(), 'percentual':p.isna().mean().mul(100)})
print(qualidade)
medidos = p.dropna(subset=['body_mass_g']).copy()
print('Observações com massa:', len(medidos))

enriquecido = p.assign(body_mass_kg=p['body_mass_g']/1000)
print(enriquecido[['body_mass_g','body_mass_kg']].head())
print(p['species'].astype('string').str.lower().head())

### Enxergando o agrupamento em três passos

Considere a pergunta “qual é a massa média de cada espécie?”. Primeiro escolha a chave: species. Depois escolha a medida: body_mass_g. Por fim escolha o cálculo: mean. A expressão é `p.groupby('species')['body_mass_g'].mean()`. O resultado tem uma entrada por espécie, em vez de uma por animal. O groupby sozinho prepara o agrupamento; a média é calculada na chamada mean.

O resultado esperado para Adelie é aproximadamente 3700,66 g, baseado em 151 massas presentes, embora existam 152 observações da espécie. mean ignora as massas ausentes. Por isso a média deve vir acompanhada de count. A mediana, chamada median, é o valor central dos valores ordenados; std mede dispersão, com desvio padrão amostral por padrão. Não interprete std como erro da média.

No exemplo guiado, `massa_media_g=('body_mass_g', 'mean')` significa: “chame a saída de massa_media_g; use a coluna body_mass_g; aplique mean”. as_index=False mantém a chave como uma coluna comum do relatório. Para um resumo, use agg. Para anexar a média da espécie a cada animal, use transform: ele repete a média apropriada em cada linha, preservando o alinhamento. Subtrair essa média da massa mostra quanto cada animal está acima ou abaixo da média de seu grupo.

### Aprofundamento e cuidados

Agrupar significa separar observações por uma ou mais chaves e calcular um resumo de cada grupo. `agg` reduz os grupos; `transform` devolve um resultado alinhado às linhas originais. Use agregações nomeadas para dar nomes claros ao relatório.

`size` conta linhas; `count` conta valores presentes em uma coluna; `nunique` conta valores distintos, excluindo ausências por padrão. Essas perguntas são diferentes. Em grupos por sexo, `dropna=False` preserva o grupo sem informação. `observed=True` explicita que categorias não observadas não precisam aparecer.

Médias sem tamanho de amostra escondem fragilidade. Publique ao menos número de observações e número de medições válidas. Uma diferença entre espécies não demonstra causalidade. Correlação entre massa e nadadeira também não prova que uma medida causa a outra.


## Exemplo guiado

In [ ]:
resumo = p.groupby('species', as_index=False, observed=True).agg(
    observacoes=('body_mass_g','size'),
    massas_validas=('body_mass_g','count'),
    massa_media_g=('body_mass_g','mean'))
print(resumo)

## Exercício 05.1
Conte animais por espécie.

In [ ]:
# Escreva sua resposta aqui


## Exercício 05.2
Calcule mediana e desvio padrão da massa por espécie.

In [ ]:
# Escreva sua resposta aqui


## Exercício 05.3
Conte animais por ilha e sexo, incluindo sexo ausente.

In [ ]:
# Escreva sua resposta aqui


## Exercício 05.4
Crie uma coluna com a média de massa da espécie de cada animal e outra com seu desvio dessa média.

In [ ]:
# Escreva sua resposta aqui


## Exercício 05.5
Compare size e count para a massa por espécie. Interprete diferenças.

In [ ]:
# Escreva sua resposta aqui
